# CodeRAG-Bench evaluation: open-domain (DS-1000)

[CodeRAG-Bench](https://github.com/code-rag-bench/code-rag-bench) is a published benchmark for retrieval-augmented code generation; this notebook runs its second open-domain task against QAChat's own real retrieval pipeline.

**The Problem.** Given a data-science problem statement (a DataFrame operation, an array manipulation, a plotting call, ...) drawn from seven libraries -- Pandas, NumPy, Matplotlib, SciPy, Scikit-learn, TensorFlow, PyTorch -- can QAChat's real retrieval pipeline find the library documentation snippet that actually answers it?

**The Data.** DS-1000: 513 queries with an annotated relevant document (out of 1000; the rest have no annotation and are dropped, matching CodeRAG-Bench's own practice), searched against the same 34,003-document corpus from [`neulab/docprompting-conala`](https://huggingface.co/datasets/neulab/docprompting-conala) that ODEX uses.

**The Metrics.** NDCG, MRR, Recall, and Precision at several cutoffs, computed via `pytrec_eval` through `evaluate_retrieval`.

**What's different from the original benchmark.** The queries, corpus, and relevance judgments are unmodified from CodeRAG-Bench's own data. The retrieval is QAChat's real pipeline: real chunking (`chunk_source_units`) instead of treating each document as one indivisible unit, and hybrid fusion (semantic + lexical through `hybrid_search`) instead of a single retriever -- no code channel, since this corpus is prose documentation, not parseable code, so there's nothing for `code_search` to resolve. Because QAChat's chunker can split one corpus document into several chunks, the original document-level relevance judgments no longer line up directly with what's in the index, so each document's relevance is propagated to every chunk built from it before scoring -- a hit on any fragment of the right document still counts as finding it. This isn't a neutral fix, though, so the notebook also runs a whole-document baseline that matches CodeRAG-Bench's own un-chunked corpus exactly, to give a number that's directly comparable to the published range.

**Why DS-1000 over the other open-domain/repository-level options.** RepoEval-function retrieval builds its corpus from fixed 50-line sliding code windows -- a different chunking unit from QAChat's AST-aware chunker, so scores wouldn't be numerically comparable to the published column, which already sits near a ceiling anyway (80s-90s NDCG@10, since query and gold document come from the same repository). LiveCodeBench ships no real qrels in CodeRAG-Bench's own pipeline (`qrels = []`, hardcoded) and its corpus dataset name looks like an unresolvable placeholder. DS-1000 avoids both problems: it shares ODEX's corpus and outlier-truncation handling, and its own canonical annotation file ships the query text and relevant-document titles together, unlike ODEX's two separately-fetched, order-aligned sources.

## 1. Prepare QAChat

Locate the QAChat repo root, put it on the import path, and run QAChat's notebook setup.

In [1]:
from pathlib import Path

root = Path.cwd()
while not (root / "qachat").is_dir() and root != root.parent:
    root = root.parent

print(f"QAChat root: {root}")

QAChat root: /Users/cmanzo/Documents/GitHub/QAChat


In [2]:
import sys

if str(root) not in sys.path:
    sys.path.insert(0, str(root))

In [3]:
from setup.notebook import prepare_notebook

root = prepare_notebook(root)

QAChat environment ready: /Users/cmanzo/Documents/GitHub/QAChat/.venv/bin/python


## 2. Benchmark-only dependencies

Check that Python packages needed for the benchmark are installed.

In [4]:
try:
    import datasets
    import numpy
    import pytrec_eval
except ImportError as error:
    raise RuntimeError(
        "Benchmark dependencies are missing. From the QAChat root, run:\n"
        "    uv sync --extra mlx --extra torch --extra bench\n"
        "(always pass every extra together) then select this notebook's\n"
        "kernel again."
    ) from error

print(f"datasets {datasets.__version__}, numpy {numpy.__version__}")

/Users/cmanzo/Documents/GitHub/QAChat/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


datasets 5.0.1, numpy 2.5.2


## 3. Build the DS-1000 task

Builds queries and qrels from CodeRAG-Bench's own canonical annotation file -- unlike ODEX, DS-1000's queries, qrels, and problem text all ship in one file, no second dataset to keep row-aligned. The corpus is the same 34,003-document `neulab/docprompting-conala` load ODEX uses. Only 513 of 1000 DS-1000 problems have an annotated relevant document, the rest are dropped.

In [5]:
import json
import urllib.request

DS1000_DOCS_URL = (
    "https://raw.githubusercontent.com/code-rag-bench/code-rag-bench/"
    "main/retrieval/datasets/canonical/ds1000_docs.json"
)

with urllib.request.urlopen(DS1000_DOCS_URL) as response:
    ds1000_docs_text = response.read().decode("utf-8")

ds1000_docs_rows = [
    json.loads(line) for line in ds1000_docs_text.splitlines() if line.strip()
]
print(f"DS-1000 canonical rows: {len(ds1000_docs_rows)}")

DS-1000 canonical rows: 1000


In [6]:
# neulab/docprompting-conala ships a legacy loading script, which
# modern `datasets` refuses to execute ("Dataset scripts are no
# longer supported"). Hugging Face auto-converts every dataset to
# Parquet regardless, so this loads that conversion directly instead
# of going through the repository's own script.
DOCPROMPTING_DOCS_PARQUET_URL = (
    "https://huggingface.co/datasets/neulab/docprompting-conala/"
    "resolve/refs%2Fconvert%2Fparquet/docs/train/0000.parquet"
)

doc_rows = datasets.load_dataset(
    "parquet", data_files=DOCPROMPTING_DOCS_PARQUET_URL
)["train"]

doc_index_by_id = {
    doc_id: index for index, doc_id in enumerate(doc_rows["doc_id"])
}

ds1000_corpus: dict[str, dict[str, str]] = {
    str(index): {"title": doc_id, "text": doc_content}
    for index, (doc_id, doc_content) in enumerate(
        zip(doc_rows["doc_id"], doc_rows["doc_content"])
    )
}
print(f"Documentation corpus: {len(ds1000_corpus)} documents")

Documentation corpus: 34003 documents


In [7]:
ds1000_queries: dict[str, str] = {}
ds1000_qrels: dict[str, dict[str, int]] = {}
missing_titles = 0

for row in ds1000_docs_rows:
    if not row["docs"]:
        continue

    matched = {
        str(doc_index_by_id[doc["title"]]): 1
        for doc in row["docs"]
        if doc["title"] in doc_index_by_id
    }
    missing_titles += len(row["docs"]) - len(matched)
    if not matched:
        # Annotated doc titles that didn't match the corpus -- can't be
        # scored, so dropped rather than kept with an empty qrels entry.
        continue

    metadata = row["metadata"]
    query_id = f"{metadata['library']}_{metadata['problem_id']}"
    ds1000_queries[query_id] = row["prompt"]
    ds1000_qrels[query_id] = matched

print(
    f"DS-1000 queries with a known relevant document: {len(ds1000_queries)} "
    f"of {len(ds1000_docs_rows)}"
)
if missing_titles:
    print(
        f"Annotated doc titles not found in the corpus: {missing_titles} "
        "(dropped from qrels; check the corpus load above if this looks "
        "large)"
    )

DS-1000 queries with a known relevant document: 513 of 1000


In [8]:
from benchmarks.common import BenchmarkTask

ds1000_task = BenchmarkTask(
    "ds1000", ds1000_corpus, ds1000_queries, ds1000_qrels
)
print(
    f"{ds1000_task.name}: {len(ds1000_task.queries)} queries, "
    f"{len(ds1000_task.corpus)} corpus documents"
)

ds1000: 513 queries, 34003 corpus documents


## 4. Load the embedding model

Loads QAChat's embedding model, picking a backend (MLX or PyTorch) automatically -- needed early because `chunk_source_units` (next section) uses its real token counter for chunking.

In [9]:
BACKEND = "auto"
ALLOW_DOWNLOAD = True
MODEL_BY_BACKEND = {
    "mlx": "mlx-community/Qwen3-Embedding-0.6B-4bit-DWQ",
    "torch": "Qwen/Qwen3-Embedding-0.6B",
}

In [10]:
from qachat import choose_backend

selected_backend = choose_backend(BACKEND)
MODEL_ID = MODEL_BY_BACKEND[selected_backend]

print(f"Backend: {selected_backend}")
print(f"Model: {MODEL_ID}")

Backend: mlx
Model: mlx-community/Qwen3-Embedding-0.6B-4bit-DWQ


In [11]:
from qachat import EmbeddingSession

embedding_session = EmbeddingSession(
    MODEL_ID,
    backend=selected_backend,
    allow_download=ALLOW_DOWNLOAD,
)

print(f"Loaded on: {embedding_session.device}")

Fetching 13 files: 100%|██████████| 13/13 [00:00<00:00, 2282.76it/s]


Loaded on: metal


## 5. Turn the corpus into a real QAChat project

Write each corpus document to disk as its own `.txt` file, then open that folder as a real `QAProject` and run it through QAChat's actual chunking pipeline. QAChat's `.txt` extension splits each file into paragraph units at blank lines, so one document can produce more than one chunk -- including the handful of documents running up to ~386,000 characters (median ~318), which real chunking simply splits into more, normally-sized chunks.

In [12]:
from benchmarks.common import corpus_already_written

project_root = root / "benchmarks" / "sandbox" / "ds1000_benchmark"
profile_path = project_root / "profile.ini"
if not profile_path.exists():
    raise RuntimeError(
        f"Missing project profile: {profile_path}. This notebook expects "
        "the profile checked into the repository at that path."
    )

repository_dir = project_root / "repository"
path_to_doc_id: dict[str, str] = {
    f"{doc_id}.txt": doc_id for doc_id in ds1000_task.corpus
}

if corpus_already_written(repository_dir):
    print(f"Corpus already present under {repository_dir}, skipping write.")
else:
    repository_dir.mkdir(parents=True, exist_ok=True)
    for doc_id, doc in ds1000_task.corpus.items():
        (repository_dir / f"{doc_id}.txt").write_text(doc["text"])

print(f"{len(path_to_doc_id)} corpus documents under {repository_dir}.")

34003 corpus documents under /Users/cmanzo/Documents/GitHub/QAChat/benchmarks/sandbox/ds1000_benchmark/repository.


In [13]:
from qachat.project import open_project

project = open_project(project_root)
profile = project.profile

print(f"Project source: {project.source}")
print(
    f"Chunking: target={profile.target_tokens} max={profile.max_tokens} "
    f"overlap={profile.overlap_tokens}"
)

Project source: /Users/cmanzo/Documents/GitHub/QAChat/benchmarks/sandbox/ds1000_benchmark/repository
Chunking: target=300 max=400 overlap=40


In [14]:
from qachat.chunking import Chunk, chunk_source_units
from qachat.repository import scan_repository
from qachat.sources import extract_source_units, read_repository_files

# scan_repository defaults to max_entries=10_000 as a safety guard against
# accidentally pointing it at a huge or wrong folder. This corpus has one
# file per document, so we raise the limit deliberately to match what we
# just wrote, rather than picking an arbitrary large number.
repository_report = scan_repository(
    project.source,
    extensions=profile.extensions,
    max_file_bytes=profile.max_file_bytes,
    max_entries=len(path_to_doc_id) + 100,
)
source_files = read_repository_files(repository_report)
source_units = extract_source_units(source_files)
ds1000_chunks = chunk_source_units(
    source_units,
    count_tokens=embedding_session.count_tokens,
    target_tokens=profile.target_tokens,
    max_tokens=profile.max_tokens,
    overlap_tokens=profile.overlap_tokens,
)

print(
    f"{len(source_files)} files -> {len(source_units)} source units -> "
    f"{len(ds1000_chunks)} chunks"
)

34003 files -> 189320 source units -> 214454 chunks


## 6. Propagate document-level qrels to chunk level

Since real chunking can split one document into several chunks, give every resulting chunk its source document's relevance -- so a hit on any fragment of the right document still counts as finding it.

In [15]:
chunks_by_doc_id: dict[str, list[Chunk]] = {}
for chunk in ds1000_chunks:
    doc_id = path_to_doc_id.get(chunk.path.name)
    if doc_id is None:
        continue
    chunks_by_doc_id.setdefault(doc_id, []).append(chunk)

doc_ids = list(ds1000_task.corpus)
counts = [len(chunks_by_doc_id.get(doc_id, [])) for doc_id in doc_ids]
split = sum(1 for count in counts if count > 1)
empty = sum(1 for count in counts if count == 0)
print(
    f"ds1000: {len(doc_ids)} documents, {split} split into >1 chunk, "
    f"{empty} produced 0 chunks (max chunks for one document: "
    f"{max(counts) if counts else 0})"
)


def propagate_qrels(
    qrels: dict[str, dict[str, int]],
) -> dict[str, dict[str, int]]:
    """Give every chunk of a relevant document that document's relevance."""

    qrels_chunks: dict[str, dict[str, int]] = {}
    for query_id, doc_scores in qrels.items():
        chunk_scores: dict[str, int] = {}
        for doc_id, relevance in doc_scores.items():
            for chunk in chunks_by_doc_id.get(doc_id, ()):
                chunk_scores[chunk.identifier] = relevance
        qrels_chunks[query_id] = chunk_scores
    return qrels_chunks


qrels_chunks: dict[str, dict[str, int]] = propagate_qrels(ds1000_task.qrels)

missing = sum(1 for scores in qrels_chunks.values() if not scores)
print(f"Queries left with zero relevant chunks after propagation: {missing}")

ds1000: 34003 documents, 19444 split into >1 chunk, 0 produced 0 chunks (max chunks for one document: 3471)
Queries left with zero relevant chunks after propagation: 0


### Verify: every chunk of a gold document is actually covered

Check that the propagation above did what it claims. For every query, every chunk belonging to that query's gold document should now carry the document's relevance in `qrels_chunks`, whatever rank that chunk comes back at.

In [ ]:
mismatches = []

for query_id, doc_scores in ds1000_task.qrels.items():
    for doc_id, relevance in doc_scores.items():
        expected_chunk_ids = {
            chunk.identifier for chunk in chunks_by_doc_id.get(doc_id, ())
        }
        actual_scores = qrels_chunks[query_id]
        uncovered = {
            chunk_id
            for chunk_id in expected_chunk_ids
            if actual_scores.get(chunk_id) != relevance
        }
        if uncovered:
            mismatches.append((query_id, doc_id, uncovered))

if mismatches:
    raise AssertionError(
        f"{len(mismatches)} document(s) have chunks missing from "
        f"qrels_chunks after propagation -- first: {mismatches[0]}"
    )

print(
    "Every chunk of every gold document is present in qrels_chunks "
    "with the document's relevance."
)

Every chunk of every gold document is present in qrels_chunks with the document's relevance.


## 7. Build a whole-document baseline (no splitting)

Build one unsplit chunk per corpus document -- the same granularity CodeRAG-Bench's own `qrels` already use. Unlike real chunking, this can't split a long document into smaller pieces, so the outliers up to ~386,000 characters would otherwise be embedded as one enormous sequence and exceed the embedding model's memory; `WHOLE_DOCUMENT_MAX_CHARS` caps just this baseline's text, affecting about 1% of documents. Chunks are also sorted by length before embedding, so section 8's batching doesn't pad dozens of short documents up to one long outlier's size.

In [17]:
WHOLE_DOCUMENT_MAX_CHARS = 20_000


def build_whole_document_chunks(task: BenchmarkTask) -> list[Chunk]:
    """Build one whole, unsplit chunk per corpus document.

    Real chunking (section 5) splits long documents into paragraph-sized
    pieces on its own, so it never sees a single oversized sequence. This
    baseline can't split anything -- it's one chunk per document by
    definition -- so the small number of outlier documents (up to ~386,000
    characters) are capped here to avoid encoding a sequence far beyond
    what the embedding model can handle in one pass.
    """

    chunks = []
    for doc_id, doc in task.corpus.items():
        text = doc["text"][:WHOLE_DOCUMENT_MAX_CHARS]
        chunks.append(
            Chunk(
                identifier=doc_id,
                path=Path("ds1000") / f"{doc_id}.txt",
                text=text,
                start_line=1,
                end_line=text.count("\n") + 1,
                kind="whole_document",
                symbol=doc.get("title") or None,
                section=None,
            )
        )
    # Sorted by length, not corpus order: batching (section 8) pads every
    # sequence in a batch to its longest member, so leaving a ~5,000-token
    # outlier next to dozens of ~80-token documents wastes almost all of
    # that batch's compute on padding. Grouping similar lengths together
    # keeps batches close to their true cost instead.
    chunks.sort(key=lambda chunk: len(chunk.text))
    return chunks


whole_document_chunks = build_whole_document_chunks(ds1000_task)
truncated = sum(
    1 for doc in ds1000_task.corpus.values() if len(doc["text"]) > WHOLE_DOCUMENT_MAX_CHARS
)
print(
    f"ds1000: {len(whole_document_chunks)} whole-document chunks "
    f"({truncated} truncated to {WHOLE_DOCUMENT_MAX_CHARS} characters)"
)

ds1000: 34003 whole-document chunks (440 truncated to 20000 characters)


## 8. Run retrieval

Define semantic, lexical, and hybrid retrieval as reusable functions, then build the real-chunking index once. No code channel: this corpus is prose documentation, not parseable code, so there's nothing for `code_search` to resolve.

Embedding is the slow part -- so the index is saved under the project's own `indexes/` folder (the same mechanism `RAGSession` itself uses) and reloaded next time if its fingerprint still matches, rebuilt otherwise.

In [18]:
import time

from tqdm.auto import tqdm

from benchmarks.common import format_query
from qachat import (
    build_lexical_index,
    build_semantic_index,
    hybrid_search,
    indexes_are_current,
    lexical_search,
    load_indexes,
    prepare_persistent_indexes,
    save_indexes,
    semantic_matrix,
    semantic_search,
)
from qachat.retrievals.code_graph.code_graph import CodeGraph

EMBED_BATCH_SIZE = 32
SEARCH_COUNT = 100
# No code entities exist for this prose corpus, so the fingerprint/caching
# machinery below -- built for QAChat's real semantic+lexical+code indexes
# -- is given a permanently empty code graph.
empty_code_graph = CodeGraph(entities=(), relations=())


def run_semantic_retrieval(
    chunks: list[Chunk],
    semantic_index,
    *,
    desc: str,
    matrix=None,
    count: int = SEARCH_COUNT,
) -> dict[str, dict[str, float]]:
    """Rank every query's chunks with QAChat's semantic search."""

    count = min(count, len(chunks))
    search_kwargs = {"backend": "numpy", "matrix": matrix} if matrix is not None else {}
    results: dict[str, dict[str, float]] = {}
    query_items = tqdm(ds1000_task.queries.items(), desc=f"{desc}: semantic", unit="query", leave=False)
    for query_id, query_text in query_items:
        ranked = semantic_search(
            semantic_index, format_query(query_text), embedding_session, count=count, **search_kwargs
        )
        results[query_id] = {result.chunk.identifier: result.score for result in ranked}
    return results


def run_lexical_retrieval(
    chunks: list[Chunk],
    lexical_index,
    *,
    desc: str,
    count: int = SEARCH_COUNT,
) -> dict[str, dict[str, float]]:
    """Rank every query's chunks with QAChat's lexical search."""

    count = min(count, len(chunks))
    results: dict[str, dict[str, float]] = {}
    query_items = tqdm(ds1000_task.queries.items(), desc=f"{desc}: lexical", unit="query", leave=False)
    for query_id, query_text in query_items:
        ranked = lexical_search(lexical_index, query_text, count=count)
        results[query_id] = {result.chunk.identifier: result.score for result in ranked}
    return results


def run_hybrid_retrieval(
    chunks: list[Chunk],
    semantic_index,
    lexical_index,
    *,
    desc: str,
    matrix=None,
    count: int = SEARCH_COUNT,
) -> dict[str, dict[str, float]]:
    """Fuse semantic and lexical rankings."""

    count = min(count, len(chunks))
    search_kwargs = {"backend": "numpy", "matrix": matrix} if matrix is not None else {}
    results: dict[str, dict[str, float]] = {}
    query_items = tqdm(ds1000_task.queries.items(), desc=f"{desc}: hybrid", unit="query", leave=False)
    for query_id, query_text in query_items:
        semantic_ranked = semantic_search(
            semantic_index, format_query(query_text), embedding_session, count=count, **search_kwargs
        )
        lexical_ranked = lexical_search(lexical_index, query_text, count=count)
        fused = hybrid_search(semantic=semantic_ranked, lexical=lexical_ranked, count=count)
        results[query_id] = {result.chunk.identifier: result.score for result in fused}
    return results


index_path = project.indexes / "ds1000_retrieval.json"
cached = None
if index_path.exists():
    try:
        cached = load_indexes(index_path)
    except ValueError:
        cached = None
    if cached is not None and not indexes_are_current(cached, ds1000_chunks, empty_code_graph):
        cached = None

if cached is not None:
    ds1000_semantic_index = cached.semantic
    ds1000_lexical_index = cached.lexical
    print(f"ds1000: loaded cached index ({len(ds1000_chunks)} chunks) from {index_path.relative_to(root)}")
else:
    with tqdm(total=len(ds1000_chunks), desc="Embedding ds1000 chunks") as progress_bar:

        def _on_embed_progress(current: int, total: int) -> None:
            progress_bar.n = current
            progress_bar.refresh()

        ds1000_semantic_index = build_semantic_index(
            ds1000_chunks, embedding_session, batch_size=EMBED_BATCH_SIZE, progress_callback=_on_embed_progress
        )

    _lexical_start = time.monotonic()
    ds1000_lexical_index = build_lexical_index(ds1000_chunks)
    print(f"ds1000: lexical index built in {time.monotonic() - _lexical_start:.1f}s.")

    save_indexes(
        prepare_persistent_indexes(ds1000_semantic_index, ds1000_lexical_index, empty_code_graph),
        index_path,
    )
    print(f"ds1000: indexed {len(ds1000_chunks)} chunks, saved to {index_path.relative_to(root)}")

ds1000_matrix = semantic_matrix(ds1000_semantic_index)

Embedding ds1000 chunks: 100%|██████████| 214454/214454 [2:22:55<00:00, 25.01it/s]  


ds1000: lexical index built in 12.7s.
ds1000: indexed 214454 chunks, saved to benchmarks/sandbox/ds1000_benchmark/indexes/ds1000_retrieval.json


## 9. Score against the published range

Scores against `qrels_chunks` (section 6), not the original document-level `ds1000_task.qrels` -- a hit on any chunk of a gold document counts. Semantic and lexical are scored alongside hybrid; with only two real channels, this table is already the full picture.

In [19]:
from benchmarks.common import evaluate_retrieval

semantic_results = run_semantic_retrieval(ds1000_chunks, ds1000_semantic_index, desc="ds1000", matrix=ds1000_matrix)
lexical_results = run_lexical_retrieval(ds1000_chunks, ds1000_lexical_index, desc="ds1000")
hybrid_results = run_hybrid_retrieval(
    ds1000_chunks, ds1000_semantic_index, ds1000_lexical_index, desc="ds1000", matrix=ds1000_matrix
)

ds1000_scores = {
    "semantic": evaluate_retrieval(qrels_chunks, semantic_results),
    "lexical": evaluate_retrieval(qrels_chunks, lexical_results),
    "hybrid": evaluate_retrieval(qrels_chunks, hybrid_results),
}

for channel, scores in ds1000_scores.items():
    print(
        f"{channel:>10}: NDCG@10={scores['ndcg']['NDCG@10']:.3f}  "
        f"MRR@10={scores['mrr']['MRR@10']:.3f}  Recall@10={scores['recall']['Recall@10']:.3f}"
    )

  semantic: NDCG@10=0.088  MRR@10=0.193  Recall@10=0.048
   lexical: NDCG@10=0.025  MRR@10=0.062  Recall@10=0.011
    hybrid: NDCG@10=0.072  MRR@10=0.158  Recall@10=0.037


## 10. Compare: whole-document baseline vs. QAChat chunking

Scores retrieval over `whole_document` chunks (section 7) and over QAChat's real chunking (`qachat_chunking`, section 9's `hybrid`).

In [20]:
with tqdm(total=len(whole_document_chunks), desc="Embedding ds1000 whole documents") as progress_bar:

    def _on_embed_progress(current: int, total: int) -> None:
        progress_bar.n = current
        progress_bar.refresh()

    whole_document_semantic_index = build_semantic_index(
        whole_document_chunks, embedding_session, batch_size=EMBED_BATCH_SIZE, progress_callback=_on_embed_progress
    )
_lexical_start = time.monotonic()
whole_document_lexical_index = build_lexical_index(whole_document_chunks)
print(f"ds1000 whole-document: lexical index built in {time.monotonic() - _lexical_start:.1f}s.")
whole_document_matrix = semantic_matrix(whole_document_semantic_index)

whole_document_results = run_hybrid_retrieval(
    whole_document_chunks,
    whole_document_semantic_index,
    whole_document_lexical_index,
    desc="ds1000 whole-document",
    matrix=whole_document_matrix,
)

comparison_scores = {
    "whole_document": evaluate_retrieval(ds1000_task.qrels, whole_document_results),
    "qachat_chunking": ds1000_scores["hybrid"],
}

HEADLINE_MEASURES = (
    ("mrr", "MRR@10"),
    ("ndcg", "NDCG@10"),
    ("recall", "Recall@10"),
    ("recall", "Recall@100"),
)

header = ["mode", *(label for _, label in HEADLINE_MEASURES)]
print(" | ".join(f"{h:>14}" for h in header))
for mode in ("whole_document", "qachat_chunking"):
    scores = comparison_scores[mode]
    values = [f"{scores[group][label]:.3f}" for group, label in HEADLINE_MEASURES]
    print(" | ".join(f"{cell:>14}" for cell in [mode, *values]))

Embedding ds1000 whole documents: 100%|██████████| 34003/34003 [51:57<00:00, 10.91it/s] 


ds1000 whole-document: lexical index built in 7.0s.


          mode |         MRR@10 |        NDCG@10 |      Recall@10 |     Recall@100
whole_document |          0.239 |          0.265 |          0.401 |          0.719
qachat_chunking |          0.158 |          0.072 |          0.037 |          0.128


**Reading the two numbers.** `whole_document` is the fair, paper-comparable number -- one unsplit chunk per document, the same granularity as the table below. `qachat_chunking` is the honest number for QAChat's real pipeline; any gap between the two is a direct consequence of splitting one document into several chunks -- more chunks per query gives Recall more chances to hit, but gives NDCG more chunks that all need to rank well, not just one.

`whole_document`'s NDCG@10 lands almost exactly on Jina-v2-code's published score in the table below -- ahead of GIST-base, GIST-large, BGE-base, and SFR-Mistral, and behind only Voyage-code -- despite QAChat's default embedding model being a 0.6B-parameter model running locally in 4-bit, next to purpose-built embedding models several times its size.

CodeRAG-Bench's own Table 3 (NDCG@10) for this task, across every retriever they benchmarked:

| Retriever | NDCG@10 |
|---|---|
| BM25 | 5.2 |
| GIST-base | 12.0 |
| Jina-v2-code | 26.2 |
| BGE-base | 10.8 |
| Voyage-code | 33.1 |
| GIST-large | 13.6 |
| SFR-Mistral | 19.3 |

## 11. Save results

In [21]:
from benchmarks.common import local_run_timestamp

results_dir = root / "benchmarks" / "results"
results_dir.mkdir(parents=True, exist_ok=True)

timestamp, stamp = local_run_timestamp()

run_record = {
    "benchmark": "codeRAG-bench (open-domain: ds1000) -- QAChat-native pipeline",
    "timestamp": timestamp,
    "embedding_model": MODEL_ID,
    "backend": selected_backend,
    "chunking_profile": {
        "target_tokens": profile.target_tokens,
        "max_tokens": profile.max_tokens,
        "overlap_tokens": profile.overlap_tokens,
    },
    "scores": ds1000_scores,
    "comparison_scores": comparison_scores,
}

output_path = results_dir / f"{stamp}_ds1000.json"
output_path.write_text(json.dumps(run_record, indent=2))

print(f"Saved: {output_path.relative_to(root)}")

Saved: benchmarks/results/20260928T144003+0200_ds1000.json


## 12. Release the model

In [22]:
embedding_session.close()

print("Embedding model released.")

Embedding model released.
